# Paper Replicating

The goal is to turn a ML Research paper into usable code.

In this notebook we are going to be replicating vision transformer (VIT) architecture/paper  with PyTorch


## 0. Getting setup

Lets import code we have already written + required libraries

In [ ]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12 or int(torch.__version__.split(".")[0]) == 2, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py . # get the helper_functions.py script
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine
    from helper_functions import download_data, set_seeds, plot_loss_curves

In [ ]:
# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

#1. Get Data

The whole goal of what we are trying to do is to replicate the ViT architecture for our Food Vision Mini Problem.

To do that we need some data

Namely, the pizza, steak and sushi images we have been using so far.


In [ ]:
# Download pizza, steak, sushi images
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

In [ ]:
train_dir, test_dir

## 2. Create Datasets and Dataloaders

In [ ]:
from torchvision import transforms
from going_modular.going_modular import data_setup
# Create image size

IMG_SIZE = 224 # Comes from Table 3 of ViT paper.

#Create transforms pipeline

manual_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

print(f"manually created transforms : {manual_transforms}")

In [ ]:
# Create a batch size of 32
BATCH_SIZE = 32

# Create DataLoaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms,
    batch_size=BATCH_SIZE
)

len(train_dataloader), len(test_dataloader), class_names

### 2.3 Visualize a single image

In [ ]:
# Get a batch of images
image_batch, label_batch = next(iter(train_dataloader))


# Get a single image and label from the batch
image, label = image_batch[0], label_batch[0]

# View the single image and label shapes
image.shape, label

In [ ]:
# Plot the image with matplotlib
import matplotlib.pyplot as plt


plt.imshow(image.permute(1, 2, 0)) # (color_channels, height, width) -> (height, width, color_channels )
plt.title(class_names[label])
plt.axis(False);

## 3. Replicating ViT: Overview

Break down paper in smaller pieces.

* **Inputs** - What goes into the model ?
* **Outputs** - What comes out of the model/layer/block?
* **Layers** - Takes an input, manipulates it with a function (for example could be self-attention)
* **Blocks** -  A collection of layers
* **Model** - A collection of blocks


3.1 ViT overview: pieces of the puzzle

* Figure1: Visual overview of the architecture
* Four equations math equations which define the functions of each layer/block
* Different hyperparameters for the architecture/training
* Text

![](https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/08-vit-paper-figure-1-inputs-and-outputs.png)

Embedding = Learnable representation (start with random numbers improve over time)

### Four equations

![](https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/08-vit-paper-four-equations.png)

```
Model	Layers	Hidden size $D$	MLP size	Heads	Params
ViT-Base	12	768	3072	12	$86M$
ViT-Large	24	1024	4096	16	$307M$
ViT-Huge	32	1280	5120	16	$632M$
Table 1: Details of Vision Transformer model variants. Source: ViT paper.

We're going to focus on replicating ViT-Base (start small and scale up when necessary) but we'll be writing code that could easily scale up to the larger variants.

Breaking the hyperparameters down:

Layers - How many Transformer Encoder blocks are there? (each of these will contain a MSA block and MLP block)
Hidden size $D$ - This is the embedding dimension throughout the architecture, this will be the size of the vector that our image gets turned into when it gets patched and embedded. Generally, the larger the embedding dimension, the more information can be captured, the better results. However, a larger embedding comes at the cost of more computation.
MLP size - What are the number of hidden units in the MLP layers?
Heads - How many heads are there in the Multi-Head Attention layers?
Params - What are the total number of parameters of the model? Generally, more parameters leads to better performance but at the cost of more computation. You'll notice even ViT-Base has far more parameters than any other model we've used so far

## 4.1 Calculating patch embedding input and output shapes by hand

In [ ]:
# Create example values
height = 224 # H ("The training resolution is 224.")
width = 224 # W
color_channels = 3 # C
patch_size = 16 # P

# Calculate N (number of patches)
number_of_patches = int((height * width) / patch_size**2)
print(f"Number of patches (N) with image height (H={height}), width (W={width}) and patch size (P={patch_size}): {number_of_patches}")

In [ ]:
# Input shape
embedding_layer_input_shape = (height, width, color_channels)

# Output shape
embedding_layer_output_shape = (number_of_patches, patch_size**2 * color_channels)

embedding_layer_input_shape, embedding_layer_output_shape

### 4.2 Turning a single image into patches


In [ ]:
# View a single image

plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)

In [ ]:
# Get the top row of the image
image_permuted = image.permute(1,2,0) # convert image to color channels last (H,W,C)

# Index to plot the top row of pixels
patch_size = 16
plt.figure(figsize=(patch_size, patch_size))
plt.imshow(image_permuted[:patch_size,:,:])
plt.axis(False)

In [ ]:
# Setup code to plot top row as patches
img_size = 224
patch_size = 16
num_patches = int(img_size / patch_size)
print(f"Number of patches per row: {num_patches}\n Patch size: {patch_size} pixels x {patch_size} pixels")

assert img_size % patch_size == 0, "Image size must be divisible by patch size"

# Create a series of subplots

fix, axs = plt.subplots(nrows=1,
                        ncols=img_size // patch_size, # one column for each patch
                        sharex=True,
                        sharey=True,
                        figsize=(patch_size, patch_size))

# Iterate through number of patches in the top row

for i, patch in enumerate(range(0, img_size, patch_size)):
  axs[i].imshow(image_permuted[:patch_size, patch:patch+patch_size, :])
  axs[i].set_xlabel(i+1) # set the patch label
  axs[i].set_xticks([])
  axs[i].set_yticks([])


In [ ]:
# Setup code to plot whole image as patches

img_size = 224
patch_size = 16
num_patches = int(img_size / patch_size)
assert img_size % patch_size == 0, "Image size must be divisible by patch size"
print(f"""Number of patches per row: {num_patches}
Number of patches per column: {num_patches}
Ttotal patches: {num_patches*num_patches}
Patch size: {patch_size} pixels x {patch_size} pixels""")


# Create a series of subplots

fig, axs = plt.subplots(nrows=img_size // patch_size,
                        ncols=img_size // patch_size,
                        figsize=(img_size, img_size),
                        sharex=True,
                        sharey=True,)

# Loop through height and width of image
for i, patch_height in enumerate(range(0, img_size, patch_size)):
  for j, patch_width in enumerate(range(0, img_size, patch_size)):
    #Plot the permuted image on the differenct axes
    axs[i,j].imshow(image_permuted[patch_height:patch_height+patch_size, patch_width:patch_width+patch_size, :])
    # Setup label information for each subplot (patch)
    axs[i,j].set_ylabel(i+1,  rotation="horizontal",
                        horizontalalignment="right",
                        verticalalignment="center")
    axs[i,j].set_xlabel(j+1)
    axs[i,j].set_xticks([])
    axs[i,j].set_yticks([])
    axs[i,j].label_outer()

# Set up a title for the plot
fig.suptitle(f"{class_names[label]} -> Patchified", fontsize=14)

### 4.3 Creating image patches and turning them into patch embeddings

We could create the image patches and image patch embeddings in a single step using torch.nn.Conv2d() and setting the kernel size and stride parameters to patch_size

In [ ]:
# Create conv2d layer to turn image into patches of learnable feature maps (embeddings)

from torch import nn

# Set the patch size
patch_size = 16

# Create a conv2d layer with hyperparameters from the ViT paper

conv2d = nn.Conv2d(in_channels=3, # for color images
                   out_channels=768, # D size from Table 1)
                  kernel_size=patch_size,
                   stride=patch_size,
                   padding=0)
conv2d

In [ ]:
# View one image
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)

In [ ]:
# pass the image through the convolutional layer
image_out_of_conv = conv2d(image.unsqueeze(0)) # add batch dimension -> (batch_size, color_channels, height, width)
print(image_out_of_conv.shape)

### 4.4 Flattening the patch embedding with `torch.nn.Flatten()`

Right now we have a series of convolutional feature maps that we want to flatten into a sequence of patch embeddings to satisfy the input criteria of the VIT Transformer Encoder.

In [ ]:
print(f"{image_out_of_conv.shape} -> (batch_size, embedding_dim, feature_map_height, feature_map_width)")

Want: (batch_size, number_of_patches, embedding_dim)

In [ ]:
from torch import nn
flatten_layer = nn.Flatten(start_dim=2, end_dim=3)

flatten_layer(image_out_of_conv).shape

In [ ]:
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)
print(f"Original image shape: {image.shape}")

# Turn image into feature maps
image_out_of_conv = conv2d(image.unsqueeze(0)) # add batch dimension
print(f"Image feature map (patches) shape: {image_out_of_conv.shape}")

# Flatten the feature maps
image_out_of_conv_flattened = flatten_layer(image_out_of_conv)
print(f"Flattened image feature maps shape: {image_out_of_conv_flattened.shape}")

In [ ]:
# Rearrange output of flattened layer

image_out_of_conv_flattened_rearranged = image_out_of_conv_flattened.permute(0,2,1)
print(f"Rearranged image feature maps shape: {image_out_of_conv_flattened_rearranged.shape}")

In [ ]:
image_out_of_conv_flattened_permuted = image_out_of_conv_flattened.permute(0,2,1)
print(f"Rearranged image feature maps shape: {image_out_of_conv_flattened_permuted.shape} -> (batch_size, number_of_patches, embedding_dim)")

In [ ]:
#Get a single flattened feature map
single_flattened_feature_map = image_out_of_conv_flattened_permuted[:,:,0]
#Plot the flattened feature map visually
plt.figure(figsize=(22,22))
plt.imshow(single_flattened_feature_map.detach().numpy())
plt.title(f"Flattened feature map shape: {single_flattened_feature_map.shape}")
plt.axis(False);

### 4.5 Turning the ViT patch embedding layer into a Pytorch module

We want this module to do:
1. Create a class called `PatchEmbedding`
2. Initialize with appropriate hyperparameters such as channels, embedding dimension, patch size.
3. Create a layer to turn an image into embedding patches using `nn.Conv2d()`
4. Create a layer to flatten the feature maps of the output of the layer in 3.
5. Define a `forward()` that defines the forward computation (e.g. pass through layer from 3 and 4).
6. Make sure the output shape of the layer reflects the required output shape of the patch embedding.

In [ ]:
# 1. Create a class called Patch Embeddinbg
class PatchEmbedding(nn.Module):
  # 2. Initialize the layer with appropriate hyperparameters
  def __init__(self,
             in_channels:int=3,
             patch_size:int=16,
             embedding_dim:int=768):
    super().__init__()
    self.patch_size = patch_size

    # 3. Create a layer to turn an image into embedded patches
    self.patcher = nn.Conv2d(in_channels=in_channels,
                             out_channels=embedding_dim,
                             kernel_size=patch_size,
                             stride=patch_size,
                             padding=0)
    # 4. Create a layer to flatten feature map oputputs of Conv2d
    self.flatten = nn.Flatten(start_dim=2,
                              end_dim=3)

  #4. Define a forward method
  def forward(self, x):
    # Create assertion to check that inputs are the correct shape
    image_resolution = x.shape[-1]
    assert image_resolution % patch_size == 0, f"Input image size must be divisble by patch size, image shape: {image_resolution}, patch_size : {self.patch_size}"


    # 5 PErform the forward pass
    x_patched = self.patcher(x)
    x_flattened = self.flatten(x_patched)

    #6. Make the returned sequence embedding dimensions are in the right order (batch_size, number_of_patches, embedding_dimension)

    return x_flattened.permute(0,2,1)

In [ ]:
set_seeds()

# Create an instance of patch embedding layer
patchify = PatchEmbedding(in_channels=3,
                          patch_size=16,
                          embedding_dim=768)

# Pass a single image through patch embedding layer
print(f"Input image size: {image.unsqueeze(0).shape}")
patch_embedded_image = patchify(image.unsqueeze(0)) # add an extra batch dimension
print(f"Output patch embedding sequence shape: {patch_embedded_image.shape}")

In [ ]:
rand_image_tensor = torch.randn(1,3,224,224)
rand_image_tensor_bad = torch.randn(1,3,250,250)
#patchify(rand_image_tensor).shape, patchify(rand_image_tensor_bad).shape

### 4.6 Creating the class token embedding

Want to prepend a learnable class token to the start of the patch embedding.

In [ ]:
patch_embedded_image.shape

In [ ]:
# Get the batch size and embedding dimension

batch_size = patch_embedded_image.shape[0]
embedding_dimension = patch_embedded_image.shape[2]
batch_size, embedding_dimension

In [ ]:
# Create class token embedding as a learnable parameter
class_token = nn.Parameter(torch.ones(batch_size, 1, embedding_dimension), requires_grad=True)
class_token.shape

In [ ]:
patch_embedded_image.shape

In [ ]:
# Add the class token embedding to the front of the patch embedding

patch_embedded_image_with_class_embedding = torch.cat((class_token, patch_embedded_image),
                                                      dim=1) # number of patches dimension

print(patch_embedded_image_with_class_embedding)
print(f"sequence of patch embeddings with class token prepended shape: {patch_embedded_image_with_class_embedding.shape}")

### 4.7 Creating the position embedding

Want to create a series of 1D learnable position embeddings and to add them to the sequence of the patch embeddings.

In [ ]:
# View the sequence of patch embeddings with the prepended  class embedding
patch_embedded_image_with_class_embedding, patch_embedded_image_with_class_embedding.shape

In [ ]:
# Calculate N (number of patches)
number_of_patches = int((height * width) / patch_size**2)


# Get the embedding dimension
embedding_dimension = patch_embedded_image_with_class_embedding.shape[-1]

# Create the learnable 1D position embedding

position_embedding = nn.Parameter(torch.ones(1, number_of_patches+1, embedding_dimension))
position_embedding.shape

In [ ]:
# Add the position embedding to the patch and class token embedding
patch_and_position_embedding = patch_embedded_image_with_class_embedding + position_embedding
print(patch_and_position_embedding.shape)

### 4.8 Putting it all together: from image to embedding

We have written code to turn an image in a flattened sequence of patch embeddings.

Now lets do it in one cell

In [ ]:
set_seeds()

# 1. Set patch size
patch_size = 16

# 2. Print shape of original image tensor and get the image dimensions
print(f"Image tensor shape: {image.shape}")
height, width = image.shape[1], image.shape[2]

# 3. Get image tensor and add batch dimension
x = image.unsqueeze(0)
print(f"Input image with batch dimension shape: {x.shape}")

# 4. Create patch embedding layer
patch_embedding_layer = PatchEmbedding(in_channels=3,
                                       patch_size=patch_size,
                                       embedding_dim=768)

# 5. Pass image through patch embedding layer
patch_embedding = patch_embedding_layer(x)
print(f"Patching embedding shape: {patch_embedding.shape}")

# 6. Create class token embedding
batch_size = patch_embedding.shape[0]
embedding_dimension = patch_embedding.shape[-1]
class_token = nn.Parameter(torch.ones(batch_size, 1, embedding_dimension),
                           requires_grad=True) # make sure it's learnable
print(f"Class token embedding shape: {class_token.shape}")

# 7. Prepend class token embedding to patch embedding
patch_embedding_class_token = torch.cat((class_token, patch_embedding), dim=1)
print(f"Patch embedding with class token shape: {patch_embedding_class_token.shape}")

# 8. Create position embedding
number_of_patches = int((height * width) / patch_size**2)
position_embedding = nn.Parameter(torch.ones(1, number_of_patches+1, embedding_dimension),
                                  requires_grad=True) # make sure it's learnable

# 9. Add position embedding to patch embedding with class token
patch_and_position_embedding = patch_embedding_class_token + position_embedding
print(f"Patch and position embedding shape: {patch_and_position_embedding.shape}")

### Equation 2: Multiheaded attention block

- Multiheaded self attention: which part of a sequence should pay the most attention to itself?
  - In our case we have a series of embedded image patches, which patch significantly relates to another patch.
  - We want our neural network (ViT) to learn this relationship/representation.

- To replicate MSA in PyTorch we can use Multiheaded attention of pytorch
- Layer Norm = Layer normalization is a technique to normalize the distributions of intermediate layers. It enables smoother gradients faster training and better generalization accuracy.
  - Normalization make everything have the same mean and same standard deviation.
  - In PyTorch normalizes values over D dimension, in our case, the D dimension is the embedding dimesnion.
     - When we normalize along the embedding dimension, its like making all of the stairs in the staircase the same size.


In [ ]:
class MultiHeadSelfAttentionBlock(nn.Module):
  """Creates a multi head self-attention block"""
  def __init__(self,
               embedding_dim:int=768,  # Hiddlen state D
               num_heads=12, # Heads
               attn_dropout:int=0):
    super().__init__()
    #Create the norm layer (LN)
    self.layer_norm = nn.LayerNorm(normalized_shape=embedding_dim)

    #Create multiheaded attention (MSA) layer
    self.multihead_attn = nn.MultiheadAttention(embed_dim=embedding_dim,
                                                num_heads=num_heads,
                                                dropout=attn_dropout,
                                                batch_first=True) # is the batch first? (batch,seq,feature) -> (batch, number_of_patches, embedding_dimension)

  def forward(self, x):
    x = self.layer_norm(x)
    attn_output, _ = self.multihead_attn(query=x,
                                         value=x,
                                         key=x,
                                         need_weights=False)
    return attn_output

In [ ]:
#Create an instance MSA block

multihead_self_attention_block = MultiHeadSelfAttentionBlock(embedding_dim=768,
                                                             num_heads=12,
                                                             attn_dropout=0)

#Pass the patch and position image embedding sequence through MSA block.

patched_image_through_msa_block = multihead_self_attention_block(patch_and_position_embedding)
print(f"Input shape of MSA Block: {patch_and_position_embedding.shape}")
print(f"Output shape of MSA Block: {patched_image_through_msa_block.shape}")

## Equation 3: Mutlilayer Perceptron (MLP Block)


* **MLP** = The MLP contains two layers with a GELU non-linearity

In pseudocode:
```python
#MLP

x = linear -> non-linear -> dropout -> linear -> dropout
```



In [ ]:
class MLPBlock(nn.Module):
  def __init__(self,
               embedding_dim:int=768,
               mlp_size:int=3072,
               dropout:int=0.1):
    super().__init__()

    #Create the norm layer
    self.layer_norm = nn.LayerNorm(normalized_shape=embedding_dim)

    #Create the MLP
    self.mlp = nn.Sequential(
    nn.Linear(in_features=embedding_dim, out_features=mlp_size),
    nn.GELU(),
    nn.Dropout(p=dropout),
    nn.Linear(in_features=mlp_size,
              out_features=embedding_dim),
    nn.Dropout(p=dropout))

  def forward(self, x):
    x = self.layer_norm(x)
    x = self.mlp(x)
    return x
    # return self.mlp(self.layer_norm(x)) # same as above

In [ ]:
# Create an instance of MLPBlock
mlp_block = MLPBlock(embedding_dim=768,
                     mlp_size=3072,
                     dropout=0.1)

# Pass output the MSABlock through MLPBlock
patched_image_through_mlp_block = mlp_block(patched_image_through_msa_block)
print(f"Input shape of MLP Block: {patched_image_through_msa_block.shape}")
print(f"Output shape of MLP Block: {patched_image_through_mlp_block.shape}")

## 7. Creating the Transformer Encoder

The transformer encoder is a combination of alternating blocks of MSA(equation 2) and MLP (equation 3)


And there are residual connection between each block.

* Encoder = turn a sequence into learnable representation
* Deconder = from learned representation back to some sort of sequence
* Residual connections

In pseudo code
```python
# TRansformer encoder

x_input -> MSA Block -> [MSA_block_output + x_input] -> MLP block -> [MLP_block_output + MSA_block_output + x_input] --> ...

### 7.1 Create a custom Transformer encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,
               embedding_dim:int=768,
               num_heads:int=12,
               mlp_size:int=3072,
               mlp_dropout:int=0.1,
               attn_dropout:int=0):
    super().__init__()

    #Create MSA block
    self.msa_block = MultiHeadSelfAttentionBlock(embedding_dim=embedding_dim,
                                                 num_heads=num_heads,
                                                 attn_dropout=attn_dropout)
    #Create MLP block
    self.mlp_block = MLPBlock(embedding_dim=embedding_dim,
                              mlp_size=mlp_size,
                              dropout=mlp_dropout)

  def forward(self,x):
    x = self.msa_block(x) + x # residual connections for equation 2
    x = self.mlp_block(x) + x # residual connections for equation 3
    return x

In [ ]:
# Create an instance of TransformerEncoderBlock()

transformer_encoder_block = TransformerEncoderBlock()

# Get a summary using torchinfo.summary
summary(model=transformer_encoder_block,
        input_size=(1,197,768),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

### 7.2 Create a Transformer encoder layer with in-built Pytorch layers

So far we have created a transformer encoder by hand.

But because of how good the transformer architecture is PyTorch has implemented ready to use Transformer encoder layers

We can create a transformer encoder with pure PyTorch layers:L

In [ ]:
# Create the same as above with torch.nn.TransformerEncoderLayer()

torch_transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=768,
                                                              nhead=12,
                                                             dim_feedforward=3072,
                                                             dropout=0.1,
                                                             activation="gelu",
                                                             batch_first=True,
                                                             norm_first=True)


torch_transformer_encoder_layer

In [ ]:
summary(model=torch_transformer_encoder_layer,
        input_size=(1,197,768),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## 8. Putting it all together to create ViT

In [ ]:
#Create a ViT class

class ViT(nn.Module):
  def __init__(self,
               img_size:int=224,
               in_channels:int=3,
               patch_size:int=16,
               num_transformer_layers:int=12,
               embedding_dim:int=768,
               mlp_size:int=3072,
               num_heads:int=12,
               attn_dropout:int=0,
               mlp_dropout:int=0.1,
               embedding_dropout:int=0.1, # dropout for patch and position embedding
               num_classes:int=1000):
    super().__init__()

    # Make an assertion that the image size is compatible with the patch size
    assert img_size % patch_size == 0, f"Image size must be divisible by patch size, image"

    # Calculate the number of patches
    self.num_patches = (img_size * img_size) // patch_size**2


    # Create learnable class embedding (needs to go at front of sequence of patch embeddings)
    self.class_embedding = nn.Parameter(data=torch.randn(1,1, embedding_dim),
                                        requires_grad=True)

    # Create a learnable positional embedding
    self.position_embedding = nn.Parameter(data=torch.randn(1,self.num_patches+1, embedding_dim))

    # Create embedding dropout value
    self.embedding_dropout = nn.Dropout(p=embedding_dropout)


    # Create patch embedding layer
    self.patch_embedding = PatchEmbedding(in_channels=in_channels,
                                          patch_size=patch_size,
                                          embedding_dim=embedding_dim)

    # Create the Transformer Encoder block
    self.transformer_encoder = nn.Sequential(*[TransformerEncoderBlock(embedding_dim=embedding_dim,
                                                                    num_heads=num_heads,
                                                                    mlp_size=mlp_size,
                                                                    mlp_dropout=mlp_dropout) for _ in range(num_transformer_layers)])
    # Create classifier head
    self.classifier = nn.Sequential(
        nn.LayerNorm(normalized_shape=embedding_dim),
        nn.Linear(in_features=embedding_dim,
                  out_features=num_classes)
    )


  def forward(self,x):
    # Get the batch size
    batch_size = x.shape[0]
    # Create class token embedding and expand it to match the batch size
    class_token = self.class_embedding.expand(batch_size, -1, -1) # -1 mean to inner the dimensions

    # Create the patch embedding
    x = self.patch_embedding(x)

    # Concatenate the class token to the patch embedding
    x = torch.cat((class_token, x), dim=1)

    # Add the positional embedding
    x = self.position_embedding + x

    # Apply dropout to patch embedding
    x = self.embedding_dropout(x)

    # Pass position and patch embedding to transformer
    x = self.transformer_encoder(x)

    # Put 0th index logit through classifier
    x = self.classifier(x[:,0])

    return x


In [ ]:
batch_size=32
embedding_dim=768
class_embedding = nn.Parameter(data=torch.randn(1,1, embedding_dim),
                                        requires_grad=True)
class_token = class_embedding.expand(batch_size, -1, -1)
print(class_embedding.shape)
print(class_token.shape)

In [ ]:
set_seeds()

# Create a random image tensor with same shape as a single image

random_image_tensor = torch.randn(1,3,224,224)

# Create an instance of ViT with the number of classes we are working with (pizza, steak and sushi)
vit = ViT(num_classes=len(class_names))

# Pass the random image tensor to our ViT instance
vit(random_image_tensor)

###  8.1 Getting a visual summary of our ViT model





In [ ]:
from torchinfo import summary

summary(model=ViT(num_classes=len(class_names)),
        input_size=(1,3,224,224), # (batch_size, color_channels, height, width)
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## 9. Setting up training code for our custom ViT

We have replicated the ViT architecture now lets see  how it performs on our FoodVision Mini data.


### 9.1 Creating an optimizer

The paper states it uses the Adam Optimizer with B1 value of 0.9 , B2 of 0.999 (defaukts) and  a weight  decay of 0.1.

Weight decay = Weight decay is a regularization technique by adding a small penality usually wthe L2 Norm of the weights to the loss function.

Regularization technique = prevents overfitting

In [ ]:
optimizer = torch.optim.Adam(params=vit.parameters(),
                             lr=0.001,
                             betas=(0.9, 0.999),
                             eps=1e-08,
                             weight_decay=0.1)

### 9.2 Creating a loss function

The ViT paper doesnot actually mention what loss function they used.

So since its a multiclass classification we will use the torch.nn.CrossEntropyLoss().

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

### 9.3 Training our ViT model


In [ ]:
from going_modular.going_modular import engine

set_seeds()

# Create an instance of ViT
optimizer = torch.optim.Adam(params=vit.parameters(),
                             lr=0.001,
                             betas=(0.9, 0.999),
                             eps=1e-08,
                             weight_decay=0.1)

loss_fn = torch.nn.CrossEntropyLoss()

results = engine.train(model=vit,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       epochs=10,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       device=device)

## 10. Using a pretrained ViT from torchvision.models

Generally in deep learning if you can use a pretrained model from a large dataset on your own problem, its often a good place to start. If you can find a pretrained model and use transfer learning, give it a go, it often achieves great results with little data.

Why use a pretrained model ?
- Sometimes data is limited
- Limited training resources
- Get better results faster (sometimes)

In [ ]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 10.1 Prepare a pretrained ViT for use with Food vision mini

In [ ]:
# Get pretrained weights for ViT-Base
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT

# Setup a ViT model instance with pretrained weights
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights).to(device)

# Freeze the base parameters
for parameter in pretrained_vit.parameters():
    parameter.requires_grad = False

# Update the classifier head
set_seeds()
pretrained_vit.heads = torch.nn.Linear(in_features=768,
                                       out_features=len(class_names)).to(device)

In [ ]:
from torchinfo import summary
# Get a summary using torchinfo.summary
summary(model=pretrained_vit,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

### 10.3 Preparing data for the pretrained ViT model

When using a pretrained model you want to make sure your data is formatted in the same way that the model was trained on.

In [ ]:
# Get automatic transforms from pretrained ViT weights
vit_transforms = pretrained_vit_weights.transforms()
vit_transforms

In [ ]:
# Setup dataloaders
from going_modular.going_modular import data_setup

train_dataloader_pretrained, test_dataloader_pretrained, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=vit_transforms,
    batch_size=32) # Could set a higher batch size because using a pretrained model


### 10.4 Train feature extractor ViT model



In [ ]:
from going_modular.going_modular import engine

# Create optimizer and loss function
optimizer = torch.optim.Adam(params=pretrained_vit.parameters(),
                             lr=0.001)

loss_fn = torch.nn.CrossEntropyLoss()

# Train the model
set_seeds()
pretrained_vit_results = engine.train(model=pretrained_vit,
                                      train_dataloader=train_dataloader_pretrained,
                                      test_dataloader=test_dataloader_pretrained,
                                      optimizer=optimizer,
                                      loss_fn=loss_fn,
                                      epochs=10,
                                      device=device)

### 10.5 Plot the loss curves of our Pretrained ViT feature extractor model.

In [ ]:
from helper_functions import plot_loss_curves

plot_loss_curves(pretrained_vit_results)

### 10.6 Save our best performing ViT model

Now we have got a model that performs quite well, how about we save it to file and then check its filesize.

In [ ]:
# Save the model

from going_modular.going_modular import utils

utils.save_model(model=pretrained_vit,
                 target_dir="models",
                 model_name="pretrained_vit_feature_extractor_pizza_steak_sushi.pth")

In [ ]:
from pathlib import Path
pretrained_vit_model_size = Path("models/pretrained_vit_feature_extractor_pizza_steak_sushi.pth").stat().st_size // (1024 * 1024)
print(f"Pretrained ViT model size: {pretrained_vit_model_size} MB")


In [ ]:
import requests

# Import function to make predictions on images and plot them
from going_modular.going_modular.predictions import pred_and_plot_image

# Setup custom image path
custom_image_path = image_path / "04-pizza-dad.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
        # When downloading from GitHub, need to use the "raw" file link
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
        print(f"Downloading {custom_image_path}...")
        f.write(request.content)
else:
    print(f"{custom_image_path} already exists, skipping download.")

# Predict on custom image
pred_and_plot_image(model=pretrained_vit,
                    image_path=custom_image_path,
                    class_names=class_names)